# Student Notebook 05 — End-to-End Test-Set Evaluation

Run the assembled pipeline once on the held-out 257-film
test set. **This is the only notebook in the project that
touches the test set.** The number you get out of this
notebook is the headline result for your section of the
report.

**Run order:** all four previous notebooks must run first.
Re-run this notebook only if you decide to finalize a
different model in notebook 01.

**What you'll get out:** test-set AUC, ECE, conformal-style
coverage, decision-cost vs baselines, top-5 SHAP features.

## CONFIG — no knob

This notebook intentionally has no knob. The whole point of
the test set is that we touch it once with whatever the team
finalized in notebooks 01-04. If you want a different model,
change ``01_modeling`` and re-run the cascade.

In [ ]:
# The only thing to edit: which run_name's calibrated model
# do we evaluate on the test set? After the team picks one,
# paste that run_name here and run.
CONFIG = {
    "run_name": "team_baseline_rf",
}
FLOP_COST  =  50_000_000
MISS_COST  = 100_000_000
REFER_COST =      5_000

## Imports + load all four-layer artifacts

In [ ]:
# --- Paths (works from any working directory) ---
from pathlib import Path

def _find_project_root() -> Path:
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "docs" / "PROJECT_CONTEXT.md").is_file():
            return cand
    raise RuntimeError(f"Could not locate project root from {Path.cwd()!s}")

PROJECT_ROOT = _find_project_root()
DATA = PROJECT_ROOT / "data" / "processed"
STUDENT = DATA / "student" / CONFIG["run_name"]
STUDENT.mkdir(parents=True, exist_ok=True)
print(f"Project root:  {PROJECT_ROOT}")
print(f"Run name:      {CONFIG['run_name']!r}")
print(f"Artifacts go:  {STUDENT.relative_to(PROJECT_ROOT)}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, brier_score_loss, f1_score, log_loss, roc_auc_score,
)

bundle = joblib.load(STUDENT / "student_calibrated.joblib")
family = bundle["config"]["model_family"]
target_col = bundle["config"]["target"]
feat_cols = bundle["feature_columns"]
print(f"Pipeline: {family} on {target_col} | calibration={bundle['calibration_method']}")

## Test-set isolation check

Programmatic verification that test imdb_ids are disjoint
from train and cal. This is the safety guarantee.

In [ ]:
feat = pd.read_parquet(DATA / "features.parquet").reset_index()
train_ids = set(feat.loc[feat["split"] == "train", "imdb_id"])
cal_ids   = set(feat.loc[feat["split"] == "cal",   "imdb_id"])
test_ids  = set(feat.loc[feat["split"] == "test",  "imdb_id"])
assert not (train_ids & test_ids), "Leakage: train + test overlap"
assert not (cal_ids   & test_ids), "Leakage: cal + test overlap"
print(f"OK: train {len(train_ids)} | cal {len(cal_ids)} | test {len(test_ids)} (all disjoint)")

## Run the calibrated model on the test set

In [ ]:
master = pd.read_parquet(DATA / "films_joined.parquet")
df_test = feat[feat["split"] == "test"].reset_index(drop=True)
X_test = df_test[feat_cols].fillna(0).values
y_test = df_test[target_col].astype(int).values
print(f"Test set: {len(df_test)} films, positive rate {y_test.mean():.3f}")

# Calibrated probability is the headline output.
proba_test = bundle["calibrated_model"].predict_proba(X_test)[:, 1]

## Layer 1 — predictive performance on test

In [ ]:
rng = np.random.default_rng(42)
def boot_ci(metric, *args, n_boot=2000):
    arrs = [np.asarray(a) for a in args]
    n = len(arrs[0])
    point = float(metric(*arrs))
    samples = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        try:
            samples.append(float(metric(*[a[idx] for a in arrs])))
        except Exception:
            pass
    lo, hi = np.quantile(samples, [0.025, 0.975])
    return point, lo, hi

for name, fn in [
    ("AUC-ROC",  lambda y, p: roc_auc_score(y, p)),
    ("PR-AUC",   lambda y, p: average_precision_score(y, p)),
    ("F1@0.5",   lambda y, p: f1_score(y, (p >= 0.5).astype(int), zero_division=0)),
    ("log-loss", lambda y, p: log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1])),
    ("Brier",    lambda y, p: brier_score_loss(y, p)),
]:
    point, lo, hi = boot_ci(fn, y_test, proba_test)
    print(f"  {name:9s}: {point:.3f} [{lo:.3f}, {hi:.3f}]")

## Layer 2 — calibration on test

In [ ]:
def ece(y_true, y_prob, n_bins=10):
    edges = np.quantile(y_prob, np.linspace(0, 1, n_bins + 1))
    edges[0], edges[-1] = 0.0, 1.0
    edges = np.maximum.accumulate(edges)
    bin_idx = np.clip(np.digitize(y_prob, edges[1:-1]), 0, n_bins - 1)
    tot = 0.0
    for b in range(n_bins):
        mask = bin_idx == b
        if mask.any():
            tot += mask.mean() * abs(y_prob[mask].mean() - y_true[mask].mean())
    return float(tot)

print(f"  ECE on test:   {ece(y_test, proba_test):.4f}")
print(f"  Brier on test: {brier_score_loss(y_test, proba_test):.4f}")

## Layer 3 — decisions on test, system vs five baselines

In [ ]:
def decide(p, fc, mc, rc):
    costs = {"Greenlight": (1 - p) * fc, "Pass": p * mc, "Refer": rc}
    mn = min(costs.values())
    return "Refer" if costs["Refer"] == mn else min(costs, key=costs.get)

def realized(action, true, fc, mc, rc):
    if action == "Greenlight": return 0 if true == 1 else fc
    if action == "Pass":       return mc if true == 1 else 0
    return rc

sys_actions = [decide(p, FLOP_COST, MISS_COST, REFER_COST) for p in proba_test]
baselines = {
    "Always-Greenlight": ["Greenlight"] * len(y_test),
    "Always-Pass":       ["Pass"] * len(y_test),
    "Read-Everything":   ["Refer"] * len(y_test),
    "System (you!)":     sys_actions,
}
rows = []
for name, acts in baselines.items():
    t = sum(realized(a, t_, FLOP_COST, MISS_COST, REFER_COST) for a, t_ in zip(acts, y_test))
    rows.append({
        "strategy":     name,
        "total_cost_M": t / 1e6,
        "p_greenlight": np.mean(np.array(acts) == "Greenlight"),
        "p_pass":       np.mean(np.array(acts) == "Pass"),
        "p_refer":      np.mean(np.array(acts) == "Refer"),
    })
bench = pd.DataFrame(rows).sort_values("total_cost_M")
print(bench.round(3).to_string(index=False))

## Headline: per-film triage report for 5 example test films

For each film: true label, calibrated probability, action,
and the recommendation in plain English. Pick five
well-known films from the test set if you have favorites.

In [ ]:
master_lookup = master.set_index("imdb_id")["movie_name"].to_dict()
test_ids_list = df_test["imdb_id"].astype(str).tolist()

# Pick: 1 highest-prob, 1 lowest-prob, 1 closest-to-0.5, 2 random.
rng2 = np.random.default_rng(42)
idx_sorted = np.argsort(proba_test)
picks = [
    int(idx_sorted[-1]),                                  # most confident hit
    int(idx_sorted[0]),                                   # most confident flop
    int(np.argmin(np.abs(proba_test - 0.5))),             # most uncertain
    int(rng2.integers(0, len(proba_test))),               # random
    int(rng2.integers(0, len(proba_test))),               # random
]
picks = list(dict.fromkeys(picks))[:5]

for i in picks:
    iid = test_ids_list[i]
    name = master_lookup.get(iid, iid)
    label = "HIT" if y_test[i] == 1 else "FLOP"
    p = proba_test[i]
    action = sys_actions[i]
    print(f"  {name[:50]:50s}  P={p:.3f}  truth={label:>4s}  → {action}")

## Save the test-set per-film table

In [ ]:
report = pd.DataFrame({
    "imdb_id":                test_ids_list,
    "movie_name":             [master_lookup.get(i, i) for i in test_ids_list],
    "calibrated_probability": proba_test,
    "true_label":             y_test,
    "recommended_action":     sys_actions,
})
report.to_csv(STUDENT / "student_test_predictions.csv", index=False)
print(f"Saved {STUDENT / 'student_test_predictions.csv'}: {len(report)} rows")

## Compare to your teammates / the rigorous pipeline

| Quantity | Rigorous Phase 8 | Your value here |
|---|---|---|
| Test AUC roi_gt_2 | 0.507 [0.437, 0.584] | (paste yours) |
| Test ECE | 0.085 | (paste yours) |
| System total cost on test | $51.3M | (paste yours) |

Your numbers will land in this neighborhood. The fact that
the test AUC drops from the cal AUC is the honest finding
the report should lead with.

That's it. No more knob. No more re-runs on the test set.
Move on to writing the report.